In [ ]:
import os
import sys
import subprocess
import numpy as np
from IPython.display import Image, display


Note that the working directory will be set to ```main_dir``` (the base directory), not the notebooks directory. <br>
Libraries such as GPA and Force-Matching are located in ```scripts``` directory. <br>
So, change the working directory to ```os.chdir("scripts")``` or ```os.chdir("scripts/ForceMatching")``` for running GPA or Force-Matching in this notebook.

In [ ]:
lib_dir = os.getcwd()
if "notebooks" in lib_dir:
    lib_dir = os.path.join(lib_dir[:-10], 'PROFET')
elif lib_dir == "/content":
    lib_dir = lib_dir + "/GPA-source_code"
os.chdir(lib_dir)
print(lib_dir)

main_dir = os.path.dirname(lib_dir)
sys.path.insert(0, main_dir)  # for util.utils
sys.path.insert(0, lib_dir)   # for models.velocityfield
data_dir = os.path.join(main_dir, 'data', '')

os.makedirs(os.path.join(main_dir, 'data'), exist_ok=True)
os.makedirs(os.path.join(main_dir, 'assets'), exist_ok=True)

from util.utils import (
    ResourceMonitor, contrast_colors,
    reduce_dimension, save_preprocessed_data, load_preprocessed_data, visualize_data,
    generate_animation, generate_W2distance_plot,
)

## Synthetic dataset

### Data property

* 5 snapshots at different timepoints (day 0 to day 4)
* 26 genes · 1,195 total cells
* Training timepoints: `times=[0, 2, 4]`, `d_red=2`
* Cells per training tp: Day 0: 239 · Day 2: 239 · Day 4: 239
* Held-out / Intermediate: Day 1: 239 · Day 3: 239

### Data structure
Datasets are located in `data/traj_dataset` directory as `log_traj_#.log` for different days.

In [3]:
# Preprocessing function
def load_synthetic_dataset():
    import pandas as pd

    # gene_expression dataset & time information
    cls = [0, 1, 2, 3, 4]
    mats = {}
    for c in cls:
        GE_matrix_file = data_dir + 'traj_dataset/log_traj_%d.log' % c

        if os.path.exists(GE_matrix_file):
            df = pd.read_table(GE_matrix_file, sep="\t", header=None)
            #print(df)
            mats[c] = np.array(df)[:,:-1]
        else:
            print("Missing data at day %d." % c)

    print("Dimension of selected genes = ", mats[0].shape[1])

    full_matrix = np.concatenate([mats[c] for c in cls])
    time_label = [c*np.ones(mats[c].shape[0], dtype=np.int32) for c in cls]
    time_label = np.concatenate(time_label)

    return full_matrix, time_label

In [ ]:
# Data preprocess
example_name = 'synthetic' # 
d_reds = [2, 4, 8, 16,32, 64,101] # [] or list of reduced dimensions
    
full_matrix, time_label = load_synthetic_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_preprocessed_data(data, example_name)

In [5]:
from collections import Counter
cnt = Counter(time_label)

In [6]:
# Load precomputed preprocessed data
try:
    time_label, full_matrix, projected_matrix, pca = load_preprocessed_data(example_name)
except:
    time_label, full_matrix = load_preprocessed_data(example_name)

time_points = sorted(list(set(time_label)))

## PROFET

In [ ]:
# PROFET step 1: GPA
d_red = 2
times = [0, 2, 4]
dimension_reduction = 'Y'
exp_no = "test"
exp_memo = f"dim{d_red}_{exp_no}"

result_dir = os.path.join(main_dir, "assets", example_name, exp_memo)
os.makedirs(result_dir, exist_ok=True)

# tunable parameters
adjust_lr_P_epochs = 'Y'    # 'Y' to auto-adjust lr_P and epochs w.r.t. dimensionality
                            # 'N' to use specified lr_P and epochs below
lr_P = 0.0025
epochs = 5000
specify_f_Lip_threshold = 'Y'    # 'Y' to stop when f_Lip divergence < threshold
f_Lip_threshold = 1e-4

In [ ]:
os.chdir(lib_dir)

dimension_note = f'dim{d_red}-' if dimension_reduction in ['Y', 'yes'] else ''

with ResourceMonitor() as monitor:
    for i in range(len(times) - 1):
        cmd = (
            f"python3 run_GPA.py"
            f" --dataset {example_name}"
            f" --label {times[i]} {times[i+1]}"
            f" --N_dim {d_red}"
            f" --exp_no {exp_no}"
            f" --dimension_reduction {dimension_reduction}"
        )
        if adjust_lr_P_epochs in ['N', 'no']:
            cmd += f" --adjust_lr_P_epochs {adjust_lr_P_epochs} -lr_P {lr_P} --epochs {epochs}"
        if specify_f_Lip_threshold in ['Y', 'yes']:
            cmd += f" --f_Lip_threshold {f_Lip_threshold}"
        subprocess.run(cmd, shell=True, check=True)

monitor.report("PROFET GPA", dataset=example_name, d_red=d_red,
               filename=os.path.join(result_dir, "resources.txt"))

In [ ]:
# PROFET step 2: Force-Matching
intermediate_times = [1, 3]

if dimension_reduction in ['Y', 'yes']:
    dimension_note = f'dim{d_red}-'
else:
    dimension_note = ''
gpa_filenames = [
    f'KL-Lipschitz_1.0000-{times[0]}_{times[1]}times-{dimension_note}{cnt[times[1]]:04d}_{cnt[times[0]]:04d}-00-{exp_no}.pickle',
    f'KL-Lipschitz_1.0000-{times[1]}_{times[2]}times-{dimension_note}{cnt[times[2]]:04d}_{cnt[times[1]]:04d}-00-{exp_no}.pickle',
]

gpa_filenames_join = " ".join(gpa_filenames)
times_join = " ".join([str(t) for t in times])


In [ ]:
os.chdir(lib_dir)
cmd = (
    f"python3 run_ForceMatching.py"
    f" --dataset {example_name}"
    f" --ts {times_join}"
    f" --exp_memo {exp_memo}"
    f" --files {gpa_filenames_join}"
)

with ResourceMonitor() as monitor:
    subprocess.run(cmd, shell=True, check=True)

monitor.report("PROFET ForceMatching", dataset=example_name, d_red=d_red,
               filename=os.path.join(result_dir, "resources.txt"))

In [ ]:
from models.velocityfield import VelocityField

# load velocityfield
velocity_net, p = VelocityField.load(os.path.join(main_dir, "assets", example_name, exp_memo, ""))

# integrate ODE
dt = p['numerical_ts'][-1]/200
numerical_dt = dt
physical_dt = dt * p['ts'][-1] / p['numerical_ts'][-1]
input_data = full_matrix[time_label==0,:] if dimension_reduction in ['N', 'no'] else projected_matrix[time_label==0, :d_red]
X1_trpts = velocity_net.integrate(input_data, T=p['numerical_ts'][-1], dt=dt)

In [ ]:
img_src1 = os.path.join(main_dir, "assets", example_name, f"{exp_memo}-movie-particles.gif")
img_src2 = os.path.join(main_dir, "assets", example_name, f"{exp_memo}-movie-velocities.gif")
img_src3 = os.path.join(main_dir, "assets", example_name, f"{exp_memo}-w2distances.png")

generate_animation(example_name, times, intermediate_times, X1_trpts,
                   img_src1, d_red, True, colors=contrast_colors, plot_vectorfield=False)
display(Image(filename=img_src1))

generate_animation(example_name, times, intermediate_times, X1_trpts,
                   img_src2, d_red, True, colors=contrast_colors, plot_vectorfield=True)
display(Image(filename=img_src2))

generate_W2distance_plot(example_name, times, intermediate_times, X1_trpts,
                         img_src3, d_red, True, colors=contrast_colors)
display(Image(filename=img_src3))